# 9 · Compare two independent image builds

## Goal

Test whether two **unsigned** builds from the same reviewed commit produce identical RAW and ISO bytes.

## Prerequisites

Finish the standard image build and smoke test. This optional experiment runs **two more complete builds**, requires significant disk space, and cannot be performed with `SCOS_SIGNING_DIR` set.

In [ ]:
from pathlib import Path
import os, subprocess, hashlib
START=Path.cwd().resolve()
ROOT=next((p for p in (START,*START.parents) if (p/"myst.yml").is_file() and (p/"scripts/build.sh").is_file()),None)
assert ROOT is not None, "Run from the SC Linux checkout (make book-lab)"
def run(*argv):
    print("+"," ".join(map(str,argv)),flush=True)
    return subprocess.run([str(x) for x in argv],cwd=ROOT,check=True)
print("Source:",ROOT)

## Action 1 · Check prerequisites

Leave previous comparisons intact; the script deliberately refuses non-empty output directories.

In [ ]:
assert not os.environ.get("SCOS_SIGNING_DIR"), "Reproducibility requires unsigned builds"
assert (ROOT/"build/builder.id").is_file(), "Build toolchain missing"
for name in ("repro-a","repro-b"):
    folder=ROOT/"build"/name
    assert not folder.exists() or not any(folder.iterdir()), f"Move or remove previous comparison: {folder}"
print("Outputs will be written under build/repro-a/ and build/repro-b/")

## Action 2 · Build both images twice

This command builds from the committed source twice and compares complete uncompressed RAW and ISO files byte-for-byte.

In [ ]:
run("make","repro")

## Action 3 · Recheck byte equality

In [ ]:
import filecmp
for relative in ("raw/sc-linux.raw","sc-linux.iso"):
    a=ROOT/"build/repro-a"/relative
    b=ROOT/"build/repro-b"/relative
    assert filecmp.cmp(a,b,shallow=False), f"Reproducibility mismatch: {relative}"
    print("Identical:",relative)

## Check

A passed comparison establishes byte equality **for these two builds** under the same builder. It does not prove universal byte-for-byte reproducibility across hosts, availability of pinned base images, root integrity, or production security.

## Next

Record the tested commit and artifact hashes, and review [verification requirements](https://github.com/soft-cloud-dev/sc-linux/blob/main/docs/verification.md).